# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the [FAIR<sup>2</sup> Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset's Croissant schema (metadata & structure) is provided at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

We begin by loading the dataset metadata and preparing the environment.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# URL to the Croissant schema for this dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)


## 2. Data Overview

The Croissant schema describes one or more Record Sets, with each set holding fields (columns) and containing tabular data.

Let's inspect the available Record Sets, their `@id` values, and associated Fields and Columns.

In [ ]:
# List record sets, fields, and columns with their @id

from pprint import pprint

record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet'):
    # For metadata from dataverse export
    record_sets = metadata.recordSet

all_recordset_ids = []

for rs in record_sets:
    print("Record Set: {}".format(rs.id))
    all_recordset_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        print(" - Fields:")
        for field in rs.fields:
            print("     @id: {} - name: {} - data type: {}".format(field.id, getattr(field, "name", ""), getattr(field, "data_type", "")))
    if hasattr(rs, 'columns'):
        print(" - Columns:")
        for col in rs.columns:
            print("     @id: {} - name: {} - data type: {}".format(col.id, getattr(col, "name", ""), getattr(col, "data_type", "")))
    print()

# If no record sets found, print note
if not all_recordset_ids:
    print("No record sets found in this schema via `recordSet` or `record_sets`. Attempting to infer from dataset...")
    # Try to infer from possible dataset.records() yields if possible (may require the user to look at first batch)
    sample = list(dataset.records(limit=1))
    print(json.dumps(sample, indent=2))

## 3. Data Extraction

Now, let's load data from a specific record set into a pandas DataFrame.

Below, we will *choose* one or more Record Set IDs returned previously. *Replace `<RECORD_SET_ID>` below with the actual `@id` as printed in the previous step, for reproducibility.*

**Note:** For the FAIR<sup>2</sup> mass spectrometry demo dataset, the record set's `@id` is typically `'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'` or similar (see printed output above).

In [ ]:
# Specify the record sets to extract. Use their `@id` as found above.
record_sets_to_extract = [
    "https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034"    # Replace/add others as needed
]

dataframes = {}

for record_set_id in record_sets_to_extract:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Extracted columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for record set: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Let's perform common EDA steps:
- Filter by a numeric field (e.g., coverage, MW, Score, etc)
- Normalize a numeric field
- Group by a categorical value (e.g., by description, or accession)

Replace `<NUMERIC_FIELD_ID>` with the column name corresponding to the numeric field's `@id` (often, for Croissant, columns and field names are equivalent).

In [ ]:
# Parameters (replace with a valid column name or @id, as seen above)
record_set_id = "https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034"
numeric_field = "coverage"  # e.g. 'coverage' or its @id; update as printed in column listing
group_field = "description" # e.g. 'description', 'accession', ...

df = dataframes[record_set_id]

# Ensure the fields exist
if numeric_field in df.columns:
    threshold = df[numeric_field].mean()  # example: filter above mean coverage
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by group_field
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean','count'])
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
    else:
        print(f"Group field '{group_field}' not found in DataFrame columns: {df.columns.tolist()}")
else:
    print(f"Numeric field '{numeric_field}' not found in DataFrame columns: {df.columns.tolist()}")

## 5. Visualization

We can plot the distribution of a numeric field (e.g., `coverage` or `MW`) and the relationship between two fields.

_Note: Be sure the field names below match the DataFrame columns printed above._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Scatter plot: MW vs coverage if both exist
if 'MW' in df.columns and numeric_field in df.columns:
    plt.figure(figsize=(6,6))
    sns.scatterplot(data=df, x='MW', y=numeric_field)
    plt.title("MW vs {}".format(numeric_field))
    plt.xlabel("MW")
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

In this notebook, we've used `mlcroissant` to load, explore, and process the FAIR<sup>2</sup> mass spectrometry extracellular vesicle dataset. We've examined its metadata, explored its fields programmatically by their `@id`, performed basic filtering, normalization, and grouped analysis, and visualized some attributes.

Refer to the [https://mlcommons.org/croissant/](https://mlcommons.org/croissant/) documentation to further experiment with advanced data integration and provenance features.